#1: Imports and Path Setup

In [1]:
import os
import sys
import math
import json
import collections
import numpy as np

In [2]:
from google.colab import drive
try:
    drive.mount('/content/gdrive', force_remount=True)
except Exception as e:
    print("Error mounting at /content/gdrive:", e)

%cd /content/gdrive/MyDrive/Transformer_Mine

Mounted at /content/gdrive
/content/gdrive/MyDrive/Transformer_Mine


In [3]:


PROJECT_ROOT = '/content/drive/MyDrive/Transformer_Mine'
TRANSLATIONS_DIR = os.path.join(PROJECT_ROOT, 'translations')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: /content/drive/MyDrive/Transformer_Mine


#  2: BLEU Mathematical Foundation

**BLEU (Bilingual Evaluation Understudy) — Papineni et al. 2002**

BLEU measures how much the machine translation overlaps with one or more human reference translations using n-gram precision.

**Formula:**

```text
BLEU = BP * exp( sum_{n=1}^{N} w_n * log(p_n) )
```

**Where:**

* `p_n`: modified n-gram precision for n-grams of order n
* `w_n`: weight for each n-gram order (uniform: `1/N`)
* `N`: maximum n-gram order (standard: 4)
* `BP`: brevity penalty

**Modified n-gram precision:**

```text
p_n = sum_sentences( clip_count(hypothesis n-grams) )
    / sum_sentences( hypothesis n-gram count )
```

`clip_count`: each n-gram in hypothesis counted at most as many times as it appears in the reference (prevents padding with repeated common words).

**Brevity penalty:**

```text
BP = 1                    if c > r
BP = exp(1 - r/c)         if c <= r
```

* `c`: total length of all hypotheses
* `r`: total length of all references (best match length)

**Standard BLEU-4:** `N=4`, uniform weights `w_n = 0.25`

**Paper results (Table 2):**

* Transformer Base EN-DE: 27.3 BLEU
* Transformer Big EN-DE: 28.4 BLEU

**Expected result:**

* COLAB-SCALE model, Tatoeba data, 10k steps
* Expected: significantly lower than paper
* Documenting the gap is the goal — not matching it


#3: BLEU Implementation from Scratch

In [4]:
def get_ngrams(tokens, n):
    """
    Extract all n-grams from a token list.
    Args:
        tokens : list of strings
        n : n-gram order

    Returns:
        Counter of n-gram tuples
    """
    return collections.Counter(
        tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)
    )

In [5]:

def modified_precision(hypotheses, references, n):
    """
    Compute modified n-gram precision across a corpus.

    For each hypothesis, count how many of its n-grams appear
    in the reference, clipped by the reference count.
    Aggregate across the full corpus.

    Args:
        hypotheses : list of tokenised hypothesis strings (list of lists)
        references : list of tokenised reference strings (list of lists)
        n : n-gram order

    Returns:
        precision : float (0.0 if no n-grams match)
    """
    clipped_count = 0
    total_count   = 0

    for hyp, ref in zip(hypotheses, references):
        hyp_ngrams = get_ngrams(hyp, n)
        ref_ngrams = get_ngrams(ref, n)

        # Clip: each n-gram counted at most as many times as in reference
        for ngram, count in hyp_ngrams.items():
            clipped_count += min(count, ref_ngrams.get(ngram, 0))

        total_count += max(len(hyp) - n + 1, 0)

    if total_count == 0:
        return 0.0

    return clipped_count / total_count

In [6]:
def brevity_penalty(hypotheses, references):
    """
    Compute the brevity penalty.

    Penalises translations that are shorter than the reference.
    No penalty if hypothesis is longer.

    Args:
        hypotheses : list of tokenised hypotheses
        references : list of tokenised references

    Returns:
        bp : float in (0, 1]
    """
    c = sum(len(hyp) for hyp in hypotheses)
    r = sum(len(ref) for ref in references)

    if c == 0:
        return 0.0
    if c >= r:
        return 1.0
    return math.exp(1 - r / c)

In [7]:
def corpus_bleu(hypotheses, references, max_n=4, weights=None):
    """
    Compute corpus-level BLEU score.

    Args:
        hypotheses : list of hypothesis strings (raw text)
        references : list of reference strings (raw text)
        max_n : maximum n-gram order (standard: 4)
        weights : list of weights for each n-gram order
                     default: uniform (1/max_n each)

    Returns:
        bleu : BLEU score (0-100)
        stats : dict with p_n, BP, and individual scores
    """
    if weights is None:
        weights = [1.0 / max_n] * max_n

    # Tokenise by whitespace
    hyp_tokens = [h.strip().split() for h in hypotheses]
    ref_tokens = [r.strip().split() for r in references]

    # Compute modified precision for each n-gram order
    precisions = []
    for n in range(1, max_n + 1):
        p = modified_precision(hyp_tokens, ref_tokens, n)
        precisions.append(p)

    # Brevity penalty
    bp = brevity_penalty(hyp_tokens, ref_tokens)

    # BLEU score
    # If any precision is 0, BLEU is 0 (avoid log(0))
    if any(p == 0.0 for p in precisions):
        bleu = 0.0
    else:
        log_avg = sum(w * math.log(p)
                      for w, p in zip(weights, precisions))
        bleu = bp * math.exp(log_avg) * 100

    stats = {
        'BLEU' : round(bleu, 2),
        'BP' : round(bp, 4),
        'p1' : round(precisions[0] * 100, 2) if len(precisions) > 0 else 0,
        'p2' : round(precisions[1] * 100, 2) if len(precisions) > 1 else 0,
        'p3' : round(precisions[2] * 100, 2) if len(precisions) > 2 else 0,
        'p4' : round(precisions[3] * 100, 2) if len(precisions) > 3 else 0,
    }

    return bleu, stats

In [8]:
def sentence_bleu(hypothesis, reference, max_n=4):
    """
    Compute sentence-level BLEU score with smoothing.

    Uses add-1 smoothing for n-gram orders with zero count
    (Chen and Cherry 2014 smoothing method 1).

    Args:
        hypothesis : hypothesis string
        reference : reference string
        max_n : maximum n-gram order

    Returns:
        bleu : sentence BLEU score (0-100)
    """
    hyp_tokens = hypothesis.strip().split()
    ref_tokens = reference.strip().split()

    precisions = []
    for n in range(1, max_n + 1):
        hyp_ngrams = get_ngrams(hyp_tokens, n)
        ref_ngrams = get_ngrams(ref_tokens, n)

        clipped = sum(
            min(c, ref_ngrams.get(ng, 0))
            for ng, c in hyp_ngrams.items()
        )
        total = max(len(hyp_tokens) - n + 1, 0)

        # Add-1 smoothing for sentence BLEU
        precisions.append((clipped + 1) / (total + 1))

    bp = brevity_penalty([hyp_tokens], [ref_tokens])
    log_avg = sum(0.25 * math.log(p) for p in precisions)
    bleu = bp * math.exp(log_avg) * 100

    return round(bleu, 2)



# 4: Verify BLEU Implementation

Tests against known BLEU values from the original paper and common test cases.


In [9]:
# Test 1: Perfect translation — BLEU should be 100
hyp_perfect = ["the cat sat on the mat"]
ref_perfect = ["the cat sat on the mat"]
bleu, stats = corpus_bleu(hyp_perfect, ref_perfect)
print(f"\nTest 1 — perfect translation:")
print(f"  BLEU = {bleu:.2f}  (expected 100.0)")
assert abs(bleu - 100.0) < 0.01, f"Perfect BLEU wrong: {bleu}"
print("  passed")


Test 1 — perfect translation:
  BLEU = 100.00  (expected 100.0)
  passed


In [10]:
# Test 2: No overlap — BLEU should be 0
hyp_zero = ["the cat sat on the mat"]
ref_zero  = ["dogs run quickly through fields"]
bleu_zero, _ = corpus_bleu(hyp_zero, ref_zero)
print(f"\nTest 2 — no n-gram overlap:")
print(f"  BLEU = {bleu_zero:.2f}  (expected 0.0)")
assert bleu_zero == 0.0
print("  passed")


Test 2 — no n-gram overlap:
  BLEU = 0.00  (expected 0.0)
  passed


In [11]:
# Test 3: Brevity penalty — short hypothesis penalised
hyp_short = ["the cat"]
ref_long  = ["the cat sat on the mat"]
bleu_short, stats_short = corpus_bleu(hyp_short, ref_long)
print(f"\nTest 3 — short hypothesis (brevity penalty):")
print(f"  BLEU = {bleu_short:.2f}")
print(f"  BP   = {stats_short['BP']:.4f}  (expected < 1.0)")
assert stats_short['BP'] < 1.0
print("  passed")


Test 3 — short hypothesis (brevity penalty):
  BLEU = 0.00
  BP   = 0.1353  (expected < 1.0)
  passed


In [12]:
# Test 4: Papineni et al. 2002 example
# "the the the the the the the" vs "the cat is on the mat"
hyp_rep = ["the the the the the the the"]
ref_rep  = ["the cat is on the mat"]
bleu_rep, stats_rep = corpus_bleu(hyp_rep, ref_rep)
print(f"\nTest 4 — repeated word hypothesis:")
print(f"  BLEU = {bleu_rep:.2f}")
print(f"  p1   = {stats_rep['p1']:.2f}  (clipping prevents gaming)")
assert stats_rep['p1'] < 30.0, "Clipping not working correctly"
print("  passed")


Test 4 — repeated word hypothesis:
  BLEU = 0.00
  p1   = 28.57  (clipping prevents gaming)
  passed


In [13]:
# Test 5: Known corpus BLEU from WMT literature
# Simple sanity: partial overlap should give intermediate score
hyp_partial = ["the cat is on the mat", "there is a cat on the mat"]
ref_partial  = ["the cat sat on the mat", "there is a cat is on the mat"]
bleu_p, stats_p = corpus_bleu(hyp_partial, ref_partial)
print(f"\nTest 5 — partial overlap:")
print(f"  BLEU = {bleu_p:.2f}")
print(f"  p1={stats_p['p1']:.1f}  p2={stats_p['p2']:.1f}  "
      f"p3={stats_p['p3']:.1f}  p4={stats_p['p4']:.1f}")
assert 0 < bleu_p < 100
print("  passed")

print("\nAll BLEU verification tests passed.")


Test 5 — partial overlap:
  BLEU = 42.07
  p1=92.3  p2=72.7  p3=44.4  p4=14.3
  passed

All BLEU verification tests passed.


#5: Load Translation Files

In [20]:
def load_translations(path):
    """Load translations from a text file, one per line."""
    with open(path, encoding='utf-8') as f:
        lines = f.readlines()
    return [str(l).strip() for l in lines if str(l).strip()]

# Resolve path using the correct mount point (/content/gdrive)
resolved_translations_dir = '/content/gdrive/MyDrive/Transformer_Mine/translations'

greedy_path = os.path.join(resolved_translations_dir, 'val_greedy.txt')
beam_path   = os.path.join(resolved_translations_dir, 'val_beam4.txt')
ref_path    = os.path.join(resolved_translations_dir, 'val_reference.txt')

greedy_translations = load_translations(greedy_path)
beam_translations   = load_translations(beam_path)
all_references      = load_translations(ref_path)

# Since there are 2000 references for 500 translations, it indicates 4 references per source sentence.
# We will use the first reference of each group of 4 to do a 1-to-1 evaluation.
references = all_references[:500]

print(f"Translation files loaded:")
print(f"  greedy      : {len(greedy_translations)} sentences")
print(f"  beam (k=4)  : {len(beam_translations)} sentences")
print(f"  references  : {len(references)} sentences (selected 1-to-1 from {len(all_references)} total)")

# Type check
for name, lst in [('greedy', greedy_translations),
                  ('beam',   beam_translations),
                  ('ref',    references)]:
    bad = [i for i, x in enumerate(lst) if not isinstance(x, str)]
    if bad:
        print(f"  WARNING: {name} has non-string elements at {bad[:5]}")
    else:
        print(f"  {name}: all elements are strings")

assert len(greedy_translations) == len(references), \
    f"Greedy count {len(greedy_translations)} != ref count {len(references)}"
assert len(beam_translations) == len(references), \
    f"Beam count {len(beam_translations)} != ref count {len(references)}"

print()
print("Sample (first 3):")
for i in range(3):
    print(f"  ref    : {references[i]}")
    print(f"  greedy : {greedy_translations[i]}")
    print(f"  beam   : {beam_translations[i]}")

Translation files loaded:
  greedy      : 500 sentences
  beam (k=4)  : 500 sentences
  references  : 500 sentences (selected 1-to-1 from 2000 total)
  greedy: all elements are strings
  beam: all elements are strings
  ref: all elements are strings

Sample (first 3):
  ref    : Sie ist gegangen.
  greedy : Sie ist gegangen .
  beam   : Sie ist gegangen .
  ref    : Wo seid ihr alle?
  greedy : Wo sind Sie ?
  beam   : Wo sind Sie ?
  ref    : Können wir das glauben?
  greedy : Können wir das glauben ?
  beam   : Können wir das glauben ?


# 6: Compute Corpus BLEU

In [21]:

# Verify inputs one more time before scoring
assert all(isinstance(h, str) for h in greedy_translations), \
    "greedy_translations contains non-strings"
assert all(isinstance(h, str) for h in beam_translations), \
    "beam_translations contains non-strings"
assert all(isinstance(r, str) for r in references), \
    "references contains non-strings"

# Greedy decoding BLEU
bleu_greedy, stats_greedy = corpus_bleu(greedy_translations, references)

print(f"\nGreedy decoding:")
print(f"  BLEU : {stats_greedy['BLEU']:.2f}")
print(f"  BP   : {stats_greedy['BP']:.4f}")
print(f"  p1   : {stats_greedy['p1']:.2f}")
print(f"  p2   : {stats_greedy['p2']:.2f}")
print(f"  p3   : {stats_greedy['p3']:.2f}")
print(f"  p4   : {stats_greedy['p4']:.2f}")

# Beam search BLEU
bleu_beam, stats_beam = corpus_bleu(beam_translations, references)

print(f"\nBeam search (k=4, alpha=0.6):")
print(f"  BLEU : {stats_beam['BLEU']:.2f}")
print(f"  BP   : {stats_beam['BP']:.4f}")
print(f"  p1   : {stats_beam['p1']:.2f}")
print(f"  p2   : {stats_beam['p2']:.2f}")
print(f"  p3   : {stats_beam['p3']:.2f}")
print(f"  p4   : {stats_beam['p4']:.2f}")

print(f"\nBeam vs Greedy improvement: "
      f"{stats_beam['BLEU'] - stats_greedy['BLEU']:+.2f} BLEU")


Greedy decoding:
  BLEU : 0.00
  BP   : 1.0000
  p1   : 39.16
  p2   : 23.22
  p3   : 8.57
  p4   : 0.00

Beam search (k=4, alpha=0.6):
  BLEU : 0.00
  BP   : 1.0000
  p1   : 39.67
  p2   : 23.56
  p3   : 8.39
  p4   : 0.00

Beam vs Greedy improvement: +0.00 BLEU


In [22]:
# Quick check
hyp = "Sie ist gegangen ."
ref = "Sie ist gegangen."

hyp_tokens = hyp.split()
ref_tokens  = ref.split()

print(f"hyp tokens : {hyp_tokens}")
print(f"ref tokens : {ref_tokens}")
print()

hyp_4grams = get_ngrams(hyp_tokens, 4)
ref_4grams = get_ngrams(ref_tokens, 4)

print(f"hyp 4-grams : {list(hyp_4grams.keys())}")
print(f"ref 4-grams : {list(ref_4grams.keys())}")
print()

matches = sum(min(c, ref_4grams.get(ng, 0))
              for ng, c in hyp_4grams.items())
print(f"Matching 4-grams : {matches}  (expected 0 due to punctuation mismatch)")

hyp tokens : ['Sie', 'ist', 'gegangen', '.']
ref tokens : ['Sie', 'ist', 'gegangen.']

hyp 4-grams : [('Sie', 'ist', 'gegangen', '.')]
ref 4-grams : []

Matching 4-grams : 0  (expected 0 due to punctuation mismatch)


In [23]:
import re

def normalise_for_bleu(text):
    """
    Space out punctuation so references and hypotheses
    are tokenised consistently.
    Matches the normalisation applied by the BPE tokenizer.
    """
    text = re.sub(r'([\.,!?;:\(\)\[\]{}\"\'\-/])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Normalise references to match hypothesis tokenisation
references_norm = [normalise_for_bleu(r) for r in references]

# Verify
print("Before/after normalisation (first 3):")
for i in range(3):
    print(f"  before : {references[i]}")
    print(f"  after  : {references_norm[i]}")
    print(f"  hyp    : {greedy_translations[i]}")
    print()

# Now recompute BLEU with normalised references
bleu_greedy, stats_greedy = corpus_bleu(greedy_translations, references_norm)
bleu_beam,   stats_beam   = corpus_bleu(beam_translations,   references_norm)

print(f"Greedy BLEU : {stats_greedy['BLEU']:.2f}")
print(f"  p1={stats_greedy['p1']:.2f}  p2={stats_greedy['p2']:.2f}  "
      f"p3={stats_greedy['p3']:.2f}  p4={stats_greedy['p4']:.2f}")
print()
print(f"Beam BLEU   : {stats_beam['BLEU']:.2f}")
print(f"  p1={stats_beam['p1']:.2f}  p2={stats_beam['p2']:.2f}  "
      f"p3={stats_beam['p3']:.2f}  p4={stats_beam['p4']:.2f}")

Before/after normalisation (first 3):
  before : Sie ist gegangen.
  after  : Sie ist gegangen .
  hyp    : Sie ist gegangen .

  before : Wo seid ihr alle?
  after  : Wo seid ihr alle ?
  hyp    : Wo sind Sie ?

  before : Können wir das glauben?
  after  : Können wir das glauben ?
  hyp    : Können wir das glauben ?

Greedy BLEU : 42.51
  p1=72.81  p2=47.32  p3=34.84  p4=29.25

Beam BLEU   : 40.81
  p1=72.92  p2=47.48  p3=33.90  p4=27.68


#7: Compare Against Paper

In [24]:
rows = [
    ("Model", "Transformer Base", "COLAB-SCALE (3L d256)"),
    ("Dataset", "WMT14 EN-DE (4.5M)", "Tatoeba EN-DE (~80k)"),
    ("Eval set", "newstest2014", "Tatoeba val (500 sent)"),
    ("Vocab size", "~37,000", "8,000"),
    ("Training steps", "100,000", "10,000"),
    ("Hardware", "8x P100 GPU", "1x Colab GPU"),
    ("Beam size", "4", "4"),
    ("Length penalty", "alpha=0.6", "alpha=0.6"),
    ("BLEU (greedy)", "N/A", f"{stats_greedy['BLEU']:.2f}"),
    ("BLEU (beam)", "27.3", f"{stats_beam['BLEU']:.2f}"),
]

print(f"\n{'Component':<22} {'Original Paper':<28} {'Our Result'}")
print("-" * 75)
for component, paper, ours in rows:
    print(f"{component:<22} {paper:<28} {ours}")

print()
print("Note on comparability:")
print("  Our BLEU is higher than the paper's 27.3 but is NOT a better result.")
print("  Tatoeba validation sentences are short and simple compared to")
print("  WMT14 newstest2014. Short sentences inflate BLEU scores because")
print("  4-grams cover a larger fraction of each sentence.")
print("  These two numbers are measured on different datasets and")
print("  cannot be directly compared.")
print()
print("Reasons for difference (documented):")
print("  1. Eval set     : Tatoeba (simple) vs newstest2014 (news domain)")
print("  2. Dataset size : 80k pairs vs 4.5M (56x smaller training set)")
print("  3. Model size   : 3L d256 vs 6L d512 (smaller capacity)")
print("  4. Training     : 10k steps vs 100k steps (10x fewer)")
print("  5. Vocab size   : 8k vs 37k (coarser subword units)")
print()
print("A fair comparison requires WMT14 training and newstest2014 evaluation.")
print("That is the target of the Transformer Base experiment (later phase).")


Component              Original Paper               Our Result
---------------------------------------------------------------------------
Model                  Transformer Base             COLAB-SCALE (3L d256)
Dataset                WMT14 EN-DE (4.5M)           Tatoeba EN-DE (~80k)
Eval set               newstest2014                 Tatoeba val (500 sent)
Vocab size             ~37,000                      8,000
Training steps         100,000                      10,000
Hardware               8x P100 GPU                  1x Colab GPU
Beam size              4                            4
Length penalty         alpha=0.6                    alpha=0.6
BLEU (greedy)          N/A                          42.51
BLEU (beam)            27.3                         40.81

Note on comparability:
  Our BLEU is higher than the paper's 27.3 but is NOT a better result.
  Tatoeba validation sentences are short and simple compared to
  WMT14 newstest2014. Short sentences inflate BLEU scores because

#8: Per-Sentence BLEU Analysis

In [25]:
sentence_bleus = []

In [26]:
for hyp, ref in zip(beam_translations, references_norm):
    s_bleu = sentence_bleu(hyp, ref)
    sentence_bleus.append(s_bleu)

sentence_bleus = np.array(sentence_bleus)

In [27]:
print("Per-sentence BLEU statistics (beam search, normalised references)")
print("-" * 50)
print(f"  Mean   : {sentence_bleus.mean():.2f}")
print(f"  Median : {np.median(sentence_bleus):.2f}")
print(f"  Std    : {sentence_bleus.std():.2f}")
print(f"  Min    : {sentence_bleus.min():.2f}")
print(f"  Max    : {sentence_bleus.max():.2f}")
print()

# Distribution
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
print("BLEU score distribution:")
for i in range(len(bins) - 1):
    lo, hi  = bins[i], bins[i+1]
    count   = int(((sentence_bleus >= lo) & (sentence_bleus < hi)).sum())
    bar     = '#' * (count // max(1, len(sentence_bleus) // 40))
    print(f"  {lo:>3}-{hi:<3} : {count:>4}  {bar}")

print()

# Best translations
n_show   = 5
best_idx = np.argsort(sentence_bleus)[-n_show:][::-1]
print(f"Top {n_show} translations (highest sentence BLEU):")
for idx in best_idx:
    print(f"  [{idx}] BLEU={sentence_bleus[idx]:.1f}")
    print(f"    ref  : {references_norm[idx]}")
    print(f"    pred : {beam_translations[idx]}")

print()

# Worst translations
worst_idx = np.argsort(sentence_bleus)[:n_show]
print(f"Bottom {n_show} translations (lowest sentence BLEU):")
for idx in worst_idx:
    print(f"  [{idx}] BLEU={sentence_bleus[idx]:.1f}")
    print(f"    ref  : {references_norm[idx]}")
    print(f"    pred : {beam_translations[idx]}")

Per-sentence BLEU statistics (beam search, normalised references)
--------------------------------------------------
  Mean   : 58.68
  Median : 50.81
  Std    : 26.29
  Min    : 16.97
  Max    : 100.00

BLEU score distribution:
    0-10  :    0  
   10-20  :    1  
   20-30  :   24  ##
   30-40  :  154  ############
   40-50  :   41  ###
   50-60  :  125  ##########
   60-70  :    6  
   70-80  :   23  #
   80-90  :    0  
   90-100 :    0  

Top 5 translations (highest sentence BLEU):
  [0] BLEU=100.0
    ref  : Sie ist gegangen .
    pred : Sie ist gegangen .
  [2] BLEU=100.0
    ref  : Können wir das glauben ?
    pred : Können wir das glauben ?
  [18] BLEU=100.0
    ref  : Tom ist unschuldig .
    pred : Tom ist unschuldig .
  [20] BLEU=100.0
    ref  : Tom war unfreundlich .
    pred : Tom war unfreundlich .
  [22] BLEU=100.0
    ref  : Tom fehlt .
    pred : Tom fehlt .

Bottom 5 translations (lowest sentence BLEU):
  [191] BLEU=17.0
    ref  : Ich werd ' s niemandem sagen .
   

#9: Save BLEU Results

In [28]:
import csv


In [29]:

bleu_results = {
    'experiment' : 'small_scale_01',
    'model' : 'COLAB-SCALE (3L, d256, h4, ff1024)',
    'dataset' : 'Stage B — Tatoeba EN-DE validation (500 sentences)',
    'n_sentences' : len(references_norm),
    'normalisation' : 'punctuation spaced out to match BPE tokenizer output',
    'greedy' : stats_greedy,
    'beam_k4' : stats_beam,
    'paper_base' : {
        'BLEU' : 27.3,
        'dataset' : 'WMT14 newstest2014',
        'beam' : 4,
    },
    'comparability_note': (
        'Our BLEU is measured on Tatoeba (short, simple sentences). '
        'Paper BLEU is on WMT14 newstest2014 (news domain). '
        'Scores are not directly comparable. '
        'A fair comparison requires WMT14 training and newstest2014 evaluation.'
    ),
}

In [31]:

# Resolve to the correct mount point project root
PROJECT_ROOT = '/content/gdrive/MyDrive/Transformer_Mine'
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

results_json = os.path.join(RESULTS_DIR, 'bleu_small_scale.json')
with open(results_json, 'w', encoding='utf-8') as f:
    json.dump(bleu_results, f, indent=2, ensure_ascii=False)
print(f"BLEU results saved: {results_json}")

# Ensure the 'experiments' folder exists
experiments_dir = os.path.join(PROJECT_ROOT, 'experiments')
os.makedirs(experiments_dir, exist_ok=True)

# Update experiments CSV
EXPERIMENTS_CSV = os.path.join(experiments_dir, 'results.csv')
FIELDNAMES = [
    'experiment_id', 'model_config', 'dataset', 'dataset_size',
    'epochs', 'steps', 'batch_size', 'd_model', 'num_heads',
    'num_layers', 'd_ff', 'dropout', 'label_smoothing',
    'learning_rate', 'warmup_steps', 'parameters', 'gpu',
    'training_time', 'validation_loss', 'BLEU', 'checkpoint', 'notes',
]

record = {
    'experiment_id' : 'bleu_eval_small_scale_01',
    'model_config'  : 'COLAB-SCALE (3L, d256, h4, ff1024)',
    'dataset'       : 'Stage B — Tatoeba EN-DE val (500 sent)',
    'dataset_size'  : len(references_norm),
    'BLEU'          : f"greedy={stats_greedy['BLEU']:.2f}  "
                      f"beam4={stats_beam['BLEU']:.2f}",
    'notes'         : (
        f"Greedy BLEU={stats_greedy['BLEU']:.2f}, "
        f"Beam BLEU={stats_beam['BLEU']:.2f}. "
        f"Evaluated on Tatoeba val — NOT comparable to paper's "
        f"27.3 on WMT14 newstest2014. "
        f"Greedy outperforms beam on short simple sentences."
    ),
}

# If file doesn't exist, we write headers first
file_exists = os.path.exists(EXPERIMENTS_CSV)
with open(EXPERIMENTS_CSV, 'a', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    if not file_exists:
        writer.writeheader()
    writer.writerow({k: record.get(k, '') for k in FIELDNAMES})

print(f"Results logged to: {EXPERIMENTS_CSV}")
print()
print("BLEU evaluation summary:")
print(f"  Greedy BLEU : {stats_greedy['BLEU']:.2f}")
print(f"  Beam BLEU   : {stats_beam['BLEU']:.2f}")
print(f"  Paper BLEU  : 27.3  (Transformer Base, WMT14 newstest2014)")
print()
print("These scores are not directly comparable — different eval sets.")
print("WMT14 newstest2014 evaluation is the target of a later phase.")

BLEU results saved: /content/gdrive/MyDrive/Transformer_Mine/results/bleu_small_scale.json
Results logged to: /content/gdrive/MyDrive/Transformer_Mine/experiments/results.csv

BLEU evaluation summary:
  Greedy BLEU : 42.51
  Beam BLEU   : 40.81
  Paper BLEU  : 27.3  (Transformer Base, WMT14 newstest2014)

These scores are not directly comparable — different eval sets.
WMT14 newstest2014 evaluation is the target of a later phase.


#10: Save BLEU Utilities to Drive

In [32]:
import re

In [33]:
BLEU_PY = os.path.join(PROJECT_ROOT, 'bleu.py')

In [35]:


code = '''"""
bleu.py - BLEU Evaluation from Scratch
Transformer Reproduction Project - Phase 15

Usage:
    import sys
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Transformer_Mine
    from bleu import corpus_bleu, sentence_bleu, normalise_for_bleu
"""

import re
import math
import collections


def normalise_for_bleu(text):
    """Space out punctuation to match BPE tokenizer output."""
    text = re.sub(r\'([\\.,!?;:\\(\\)\\[\\]{}\\"\\'\\-/])\', r\' \\1 \', text)
    text = re.sub(r\'\\s+\', \' \', text)
    return text.strip()


def get_ngrams(tokens, n):
    return collections.Counter(
        tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)
    )


def modified_precision(hypotheses, references, n):
    clipped_count = 0
    total_count   = 0
    for hyp, ref in zip(hypotheses, references):
        hyp_ngrams = get_ngrams(hyp, n)
        ref_ngrams = get_ngrams(ref, n)
        for ngram, count in hyp_ngrams.items():
            clipped_count += min(count, ref_ngrams.get(ngram, 0))
        total_count += max(len(hyp) - n + 1, 0)
    if total_count == 0:
        return 0.0
    return clipped_count / total_count


def brevity_penalty(hypotheses, references):
    c = sum(len(h) for h in hypotheses)
    r = sum(len(r) for r in references)
    if c == 0:
        return 0.0
    if c >= r:
        return 1.0
    return math.exp(1 - r / c)


def corpus_bleu(hypotheses, references, max_n=4, weights=None):
    if weights is None:
        weights = [1.0 / max_n] * max_n
    hyp_tokens = [h.strip().split() for h in hypotheses]
    ref_tokens = [r.strip().split() for r in references]
    precisions = [modified_precision(hyp_tokens, ref_tokens, n)
                  for n in range(1, max_n + 1)]
    bp   = brevity_penalty(hyp_tokens, ref_tokens)
    if any(p == 0.0 for p in precisions):
        bleu = 0.0
    else:
        bleu = bp * math.exp(
            sum(w * math.log(p) for w, p in zip(weights, precisions))
        ) * 100
    stats = {
        \'BLEU\': round(bleu, 2),
        \'BP\'  : round(bp, 4),
        \'p1\'  : round(precisions[0] * 100, 2),
        \'p2\'  : round(precisions[1] * 100, 2),
        \'p3\'  : round(precisions[2] * 100, 2),
        \'p4\'  : round(precisions[3] * 100, 2),
    }
    return bleu, stats


def sentence_bleu(hypothesis, reference, max_n=4):
    hyp_tokens = hypothesis.strip().split()
    ref_tokens = reference.strip().split()
    precisions = []
    for n in range(1, max_n + 1):
        hyp_ngrams = get_ngrams(hyp_tokens, n)
        ref_ngrams = get_ngrams(ref_tokens, n)
        clipped    = sum(min(c, ref_ngrams.get(ng, 0))
                         for ng, c in hyp_ngrams.items())
        total      = max(len(hyp_tokens) - n + 1, 0)
        precisions.append((clipped + 1) / (total + 1))
    bp      = brevity_penalty([hyp_tokens], [ref_tokens])
    log_avg = sum(0.25 * math.log(p) for p in precisions)
    return round(bp * math.exp(log_avg) * 100, 2)
'''

with open(BLEU_PY, 'w', encoding='utf-8') as f:
    f.write(code)

print(f"bleu.py saved to: {BLEU_PY}")

bleu.py saved to: /content/gdrive/MyDrive/Transformer_Mine/bleu.py
